In [4]:
#lOAD PACKAGES
import pandas as pd
import matplotlib.pyplot as plt
import os
import sys
import plotly.express as px
import plotly.graph_objects as go  # Add this line
from plotly.subplots import make_subplots
# Import the processing module from the same folder
sys.path.append(os.path.join("..", "scripts", "analysis"))
from processing import load_solutions, add_kwargs_as_indices, combine_solutions, read_parquet_and_convert, add_fields,apply_conservative_classification


In [ ]:
ss = [
    {'solution_folder': f"RTS-GMLC_v2.1", 'net_demand_type': 'net_load'},

    
]
demand= []
random_demand = []
reserve = []
# energy_reserve = []
for s in ss:
    demand_ = pd.read_csv(os.path.join("..", "input", s['solution_folder'], 'uc','Demand.csv'))
    random_demand_ = pd.read_csv(os.path.join("..", "input", s['solution_folder'], 'ed','random_demand.csv'))
    reserve_ = pd.read_csv(os.path.join("..", "input", s['solution_folder'], 'uc','Reserve.csv'))
    demand_  = add_fields(demand_, net_demand_type = s['net_demand_type'])
    random_demand_  = add_fields(random_demand_, net_demand_type = s['net_demand_type'])
    reserve_  = add_fields(reserve_, net_demand_type = s['net_demand_type'])
    # energy_reserve_ = pd.read_csv(os.path.join("..", "input", input_file, 'uc','Energy reserve.csv')) 
    demand.append(demand_)
    random_demand.append(random_demand_)
    reserve.append(reserve_)
    # energy_reserve.append(energy_reserve_)

demand = pd.concat(demand).set_index(['net_demand_type', 'day', 'hour']).sort_index()
random_demand = pd.concat(random_demand).set_index([ 'net_demand_type', 'day', 'hour']).sort_index()
reserve = pd.concat(reserve).set_index(['net_demand_type', 'day', 'hour']).sort_index()

# random_demand = filter_demand(demand, random_demand, reserve)

imbalance = random_demand.sub(demand['demand'], axis=0, level=['net_demand_type', 'day','hour'])


In [7]:
read_files = False
write_files = False
solution_keys = ['demand','reserve', 'energy_reserve']

if not read_files:
    ss = [
        # {'solution_folder': f"RTS-GMLC_v18.3s", 'model_type' : 'envelope'},
        # {'solution_folder': f"RTS-GMLC_v19.4s", 'model_type' : 'e-reserve'},
        {'solution_folder': f"RTS-GMLC_v32.3s", 'model_type' : 'envelope'},
        {'solution_folder': f"RTS-GMLC_v32.1s", 'model_type' : 'e-reserve'},
    ]
    days = range(1,2)
    s_uc = []
    s_ed = []
    gcd_KPI_adequacy = []
    gcdi_KPI_adequacy = []
    

    for sol in ss:
        # ρ = sol['ρ']
        s = sol['solution_folder']
        # s_uc_name = 's_uc' if sol['model_type'] == 'stochastic' else 's_uc'
        # s_ed_name = 's_sed'
        s_uc_ = load_solutions("s_uc", os.path.join("..", "output", s), days, solution_keys = solution_keys,  model_type = sol['model_type'], solution_id = s)
        if sol['model_type'] != 'stochastic':
            s_ed_ = load_solutions("s_ed", os.path.join("..", "output", s), days, solution_keys = solution_keys, model_type = sol['model_type'], solution_id = s)
        else:
            s_ed_ = load_solutions("s_suc", os.path.join("..", "output", s), days, solution_keys = solution_keys, model_type = sol['model_type'], solution_id = s)
        s_uc.append(s_uc_)
        s_ed.append(s_ed_)

        # gcd_KPI_adequacy_ = read_parquet_and_convert( os.path.join("..", "output", s, "all_gcd_KPI_adequacy.parquet"))
        # gcd_KPI_adequacy_ = add_fields(gcd_KPI_adequacy_, model_type = sol['model_type'], ρ=ρ, solution_id = s) 

        # gcdi_KPI_adequacy_ = read_parquet_and_convert( os.path.join("..", "output", s, "all_gcdi_KPI_adequacy.parquet"))
        # gcdi_KPI_adequacy_ = add_fields(gcdi_KPI_adequacy_, model_type = sol['model_type'], ρ=ρ, solution_id = s)

        # gcd_KPI_adequacy.append(gcd_KPI_adequacy_)
        # gcdi_KPI_adequacy.append(gcdi_KPI_adequacy_)

    s_uc = combine_solutions(s_uc)
    s_ed = combine_solutions(s_ed)
    # gcd_KPI_adequacy = pd.concat(gcd_KPI_adequacy)
    # gcdi_KPI_adequacy = pd.concat(gcdi_KPI_adequacy)

    for k,v in s_uc.items():
        if 'µ' in v.columns:
            s_uc[k]['model_type'] =  v.apply(lambda x: 'conservative' if (x['model_type'] == 'envelope') & (x['µ'] == 1) else x['model_type'], axis=1)
    for k,v in s_ed.items():
        if 'µ' in v.columns:
            s_ed[k]['model_type'] = v.apply(lambda x: 'conservative' if (x['model_type'] == 'envelope') & (x['µ'] == 1) else x['model_type'], axis=1)
    # if 'µ' in gcdi_KPI_adequacy.columns: 
    #     gcdi_KPI_adequacy['model_type'] = gcdi_KPI_adequacy.apply(lambda x: 'conservative' if (x['model_type'] == 'envelope') & (x['µ'] == 1) else x['model_type'], axis=1)
    #     gcd_KPI_adequacy['model_type'] = gcd_KPI_adequacy.apply(lambda x: 'conservative' if (x['model_type'] == 'envelope') & (x['µ'] == 1) else x['model_type'], axis=1)
    if write_files:
        for k,v in s_uc.items():
            v.to_csv(f's_uc_{k}.csv', index=False)
        for k,v in s_ed.items():
            v.to_csv(f's_ed_{k}.csv', index=False)

else:
    s_uc = {}
    s_ed = {}
    for solution_key in solution_keys:
        s_uc[solution_key] = pd.read_csv(f's_uc_{solution_key}.csv')
        # s_ed[solution_key] = pd.read_csv(f's_ed_{solution_key}.csv')    

    # gcd_KPI_adequacy = pd.read_csv('gcd_KPI_adequacy.csv', index_col=0)
    # gcdi_KPI_adequacy = pd.read_csv('gcdi_KPI_adequacy.csv', index_col=0)







In [15]:
s_uc['reserve']['model_type'].unique()

array(['envelope'], dtype=object)

In [13]:
s_ed['demand']

,hour,demand_MW,r_id,resource,LOL_MW,LGEN_MW,iteration,day,configuration,µ,model_type,solution_id
0,1,800.961098,None,system,0.0,0.000000,demand_1,1,base_ramp_storage_envelopes_mu_1,mu_1,envelope,RTS-GMLC_v32.3s
1,2,751.592509,None,system,0.0,0.000000,demand_1,1,base_ramp_storage_envelopes_mu_1,mu_1,envelope,RTS-GMLC_v32.3s
2,3,811.027625,None,system,0.0,0.000000,demand_1,1,base_ramp_storage_envelopes_mu_1,mu_1,envelope,RTS-GMLC_v32.3s
3,4,811.472114,None,system,0.0,0.000000,demand_1,1,base_ramp_storage_envelopes_mu_1,mu_1,envelope,RTS-GMLC_v32.3s
4,5,870.674819,None,system,0.0,22.604868,demand_1,1,base_ramp_storage_envelopes_mu_1,mu_1,envelope,RTS-GMLC_v32.3s
...,...,...,...,...,...,...,...,...,...,...,...,...
19,20,3353.436483,None,system,0.0,0.000000,demand_1,1,base_ramp_storage_envelopes_up_1_dn_1,1.0,e-reserve,RTS-GMLC_v32.1s
20,21,3098.238081,None,system,0.0,0.000000,demand_1,1,base_ramp_storage_envelopes_up_1_dn_1,1.0,e-reserve,RTS-GMLC_v32.1s
21,22,2792.303357,None,system,0.0,0.000000,demand_1,1,base_ramp_storage_envelopes_up_1_dn_1,1.0,e-reserve,RTS-GMLC_v32.1s
22,23,2678.503522,None,system,0.0,0.000000,demand_1,1,base_ramp_storage_envelopes_up_1_dn_1,1.0,e-reserve,RTS-GMLC_v32.1s


In [ ]:
imbalance